# GS 2026.2 — Machine Learning & Modelling — CitizenTech

**Integrantes (nome completo + RM):**
- TODO

**Problema:** TODO · **Público:** TODO · **Fonte dos dados:** TODO · **ODS:** TODO

## 0. Configuração

In [ ]:
ARQUIVO = "dados/base.csv"      # TODO: base pública (Kaggle, IBGE, DataSUS, INEP, dados.gov.br...) ou simulada (documentar premissas)
ALVO = "TODO_alvo"              # TODO: coluna a prever
TIPO = "classificacao"          # "classificacao" ou "regressao"
SEMENTE = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn import metrics

CLASSIF = TIPO == "classificacao"

## 1. Entendimento e EDA
Problema, público, fonte e dicionário das variáveis: TODO

In [ ]:
df = pd.read_csv(ARQUIVO)
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe(include="all").T

In [ ]:
# Qualidade dos dados: ausentes e duplicados
print("Duplicados:", df.duplicated().sum())
df.isna().sum().sort_values(ascending=False).to_frame("ausentes")

In [ ]:
# Distribuição do alvo
if CLASSIF:
    df[ALVO].value_counts().plot(kind="bar", title=f"Distribuição de {ALVO}")
else:
    df[ALVO].plot(kind="hist", bins=30, title=f"Distribuição de {ALVO}")
plt.show()

In [ ]:
# Distribuições e correlações das variáveis numéricas
num = df.select_dtypes("number")
num.hist(figsize=(14, 10), bins=30)
plt.tight_layout(); plt.show()

corr = num.corr()
plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=90); plt.yticks(range(len(corr)), corr.columns)
plt.colorbar(); plt.title("Correlação"); plt.show()

**Análise da EDA:** TODO

## 2. Pré-processamento
Ausentes, engenharia de atributos, treino/teste e padronização dentro de `Pipeline`. O pré-processamento é ajustado só no treino, sem *data leakage*.

In [ ]:
# TODO: engenharia de atributos / remover colunas irrelevantes (ex.: IDs)
df = df.drop_duplicates()
df = df.dropna(subset=[ALVO])

X = df.drop(columns=[ALVO])
y = df[ALVO]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMENTE, stratify=y if CLASSIF else None)

col_num = X.select_dtypes("number").columns.tolist()
col_cat = X.select_dtypes(exclude="number").columns.tolist()
preproc = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), col_num),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), col_cat),
])
print("Treino:", X_train.shape, "| Teste:", X_test.shape)

## 3. Modelagem: baseline + pelo menos 2 algoritmos
Justificativa da escolha dos algoritmos: TODO

In [ ]:
if CLASSIF:
    modelos = {
        "Baseline (Dummy)": DummyClassifier(strategy="most_frequent"),
        "Regressão Logística": LogisticRegression(max_iter=1000),
        "KNN": KNeighborsClassifier(),
        "Árvore de Decisão": DecisionTreeClassifier(random_state=SEMENTE),
        "Random Forest": RandomForestClassifier(random_state=SEMENTE),
    }
    cv = StratifiedKFold(5, shuffle=True, random_state=SEMENTE)
    scoring = {"accuracy": "accuracy", "f1": "f1_macro"}
    principal = "f1"
else:
    modelos = {
        "Baseline (Dummy)": DummyRegressor(strategy="mean"),
        "Regressão Linear": LinearRegression(),
        "KNN": KNeighborsRegressor(),
        "Árvore de Decisão": DecisionTreeRegressor(random_state=SEMENTE),
        "Random Forest": RandomForestRegressor(random_state=SEMENTE),
    }
    cv = KFold(5, shuffle=True, random_state=SEMENTE)
    scoring = {"rmse": "neg_root_mean_squared_error", "r2": "r2"}
    principal = "r2"

resultados = []
for nome, est in modelos.items():
    pipe = Pipeline([("prep", preproc), ("modelo", est)])
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
    linha = {"modelo": nome}
    for m in scoring:
        linha[f"treino_{m}"] = abs(r[f"train_{m}"].mean())
        linha[f"validacao_{m}"] = abs(r[f"test_{m}"].mean())
    resultados.append(linha)
comparacao = pd.DataFrame(resultados).set_index("modelo").round(4)
comparacao

## 4. Otimização de hiperparâmetros (GridSearchCV)

In [ ]:
# TODO: ajustar o modelo e a grade conforme o melhor candidato da etapa 3
pipe_rf = Pipeline([("prep", preproc),
                    ("modelo", RandomForestClassifier(random_state=SEMENTE) if CLASSIF
                               else RandomForestRegressor(random_state=SEMENTE))])
grade = {
    "modelo__n_estimators": [100, 300],
    "modelo__max_depth": [None, 5, 10],
    "modelo__min_samples_leaf": [1, 5],
}
busca = GridSearchCV(pipe_rf, grade, cv=cv, scoring=scoring[principal], n_jobs=-1)
busca.fit(X_train, y_train)
print("Melhores parâmetros:", busca.best_params_)
print("Melhor score (validação cruzada):", round(abs(busca.best_score_), 4))
modelo_final = busca.best_estimator_

## 5. Avaliação no teste e análise de generalização

In [ ]:
pred_treino, pred_teste = modelo_final.predict(X_train), modelo_final.predict(X_test)
if CLASSIF:
    print("Acurácia treino:", round(metrics.accuracy_score(y_train, pred_treino), 4))
    print("Acurácia teste :", round(metrics.accuracy_score(y_test, pred_teste), 4))
    print(metrics.classification_report(y_test, pred_teste))
    metrics.ConfusionMatrixDisplay.from_predictions(y_test, pred_teste)
    plt.show()
else:
    for nome, yt, yp in [("treino", y_train, pred_treino), ("teste", y_test, pred_teste)]:
        print(f"{nome}: RMSE={metrics.root_mean_squared_error(yt, yp):.4f} "
              f"MAE={metrics.mean_absolute_error(yt, yp):.4f} R²={metrics.r2_score(yt, yp):.4f}")
    plt.scatter(y_test, pred_teste, alpha=0.5); plt.xlabel("real"); plt.ylabel("previsto"); plt.show()

**Overfitting/underfitting (diferença treino × teste) e justificativa do modelo final:** TODO

## 6. Interpretação: importância das variáveis

In [ ]:
nomes = modelo_final.named_steps["prep"].get_feature_names_out()
importancia = pd.Series(modelo_final.named_steps["modelo"].feature_importances_, index=nomes)
importancia.sort_values().tail(15).plot(kind="barh", title="Importância das variáveis", figsize=(8, 6))
plt.show()

**Interpretação:** TODO

## 7. Aplicação ao público definido

In [ ]:
# Previsões em exemplos não usados no treino (TODO: trocar por casos/simulações do cenário CitizenTech)
exemplos = X_test.head(5).copy()
exemplos["previsao"] = modelo_final.predict(exemplos)
exemplos["real"] = y_test.head(5).values
exemplos

**Benefício para o público:** TODO
**Limitações (dados, condições reais de uso):** TODO
**Evoluções futuras (não implementadas):** TODO